# Ozon E-CUP 2026 — Cross-Encoder V1 (rubert-tiny2)

Файнтюн лёгкого cross-encoder на **LLM-парах** (soft labels) + дообучение на ручных.

**Зачем**: тест похож на LLM-распределение. GBM на этом распределении даёт LLM-holdout 0.608 → LB 0.363 (коэффициент ~0.6). Нейросеть должна пробить потолок ручных фичей, особенно в фэшн-категориях (Обувь/Одежда/Галантерея/Ювелирка — у GBM там 0.25–0.29).

**Ориентиры для сравнения (те же сплиты, что здесь)**:
- catboost_v1 (Даня, только ручные): LLM-holdout 0.45 → LB 0.283
- catboost_our_v2 (LLM+ручные): LLM-holdout 0.608, manual-holdout 0.625 → LB 0.363

**Что нужно перед запуском**:
1. Подключить датасет с файлами: `matches.parquet`, `matches_llm.parquet`, `items_human.parquet`, **полный** `items.parquet` (4.1GB).
2. GPU (T4/P100). Одна эпоха по 11M пар ≈ 2–4 часа на T4 — укладывается в сессию Kaggle.
3. Пути в конфиге ниже поправить под свой датасет.

**Выход**: `/kaggle/working/ce_v1_final/` (модель+токенайзер) — скачать и отдать для сборки сабмита. Разрешение лицензии ок: rubert-tiny2 — MIT.

In [1]:
import os, json, re, gc, time, math
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import average_precision_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

BASE = "/kaggle/input/datasets/mihailivanovvvv/hakaton-ozon-math-items"  # <-- поправь
MATCHES_PATH = f"{BASE}/matches.parquet"
MATCHES_LLM_PATH = f"{BASE}/matches_llm.parquet"
ITEMS_PATH = f"{BASE}/items.parquet"          # полный, 4.1GB
ITEMS_HUMAN_PATH = f"{BASE}/items_human.parquet"

MODEL_NAME = "cointegrated/rubert-tiny2"   # 29M параметров, MIT, русский
MAX_LEN = 160
BATCH = 256
LR_A, LR_B = 2e-4, 5e-5     # стадия A: LLM-пары; стадия B: ручные
EPOCHS_A, EPOCHS_B = 1, 2
WARMUP = 2000
EVAL_EVERY = 10000          # шагов
CKPT = "/kaggle/working/ce_v1_ckpt.pt"
OUT_DIR = "/kaggle/working/ce_v1_final"
SEED = 42

torch.manual_seed(SEED); np.random.seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, torch.cuda.get_device_name(0) if device == "cuda" else "")

cuda Tesla T4


## 1. Текст товара
Имя + компактные атрибуты (коды, бренд, вариантные). Эту же функцию потом копируем в сабмит — менять только синхронно!

In [2]:
KEY_ORDER = ["бренд", "артикул", "партномер", "oem", "код", "модель", "размер",
             "цвет", "объем", "обьем", "вес", "тип", "материал", "количество"]

def build_text(name, attributes, max_attr_chars=260):
    parts = [str(name) if name is not None else ""]
    try:
        attrs = json.loads(attributes) if isinstance(attributes, str) else {}
    except Exception:
        attrs = {}
    if isinstance(attrs, dict) and attrs:
        low = {str(k).lower(): str(v) for k, v in attrs.items() if v}
        picked, used = [], set()
        for want in KEY_ORDER:
            for k, v in low.items():
                if want in k and k not in used:
                    picked.append(f"{k}:{v}"); used.add(k)
        rest = [f"{k}:{v}" for k, v in low.items() if k not in used]
        s = " ; ".join(picked + rest)[:max_attr_chars]
        parts.append(s)
    return " | ".join(parts)

t0 = time.time()
item_text, item_cat = {}, {}
f = pq.ParquetFile(ITEMS_PATH)
for b in f.iter_batches(columns=["id", "name", "attributes", "category"], batch_size=500_000):
    for i, n, a, c in b.to_pandas().itertuples(index=False, name=None):
        item_text[i] = build_text(n, a)
        item_cat[i] = c
print(f"items: {len(item_text):,} за {time.time()-t0:.0f}s")

items: 13,397,761 за 529s


## 2. Сплиты — ИДЕНТИЧНЫ нашим GBM-экспериментам
Групповые (union-find по товарам): manual seed 42 / 20%, LLM seed 13 / 3% компонент.

In [3]:
def group_val_mask(df, val_frac, seed):
    parent = {}
    def find(x):
        p = parent.setdefault(x, x)
        while p != parent[p]:
            parent[p] = parent[parent[p]]; p = parent[p]
        parent[x] = p; return p
    for a, b in zip(df.id1.values, df.id2.values):
        ra, rb = find(a), find(b)
        if ra != rb: parent[rb] = ra
    comp = np.fromiter((find(i) for i in df.id1.values), dtype=np.int64, count=len(df))
    rng = np.random.RandomState(seed)
    uniq = np.unique(comp)
    val_set = set(uniq[rng.rand(len(uniq)) < val_frac].tolist())
    return np.fromiter((c in val_set for c in comp), dtype=bool, count=len(df))

m = pd.read_parquet(MATCHES_PATH)
m_val_mask = group_val_mask(m, 0.20, 42)
m_train, m_val = m[~m_val_mask].copy(), m[m_val_mask].copy()

ml = pd.read_parquet(MATCHES_LLM_PATH)
l_val_mask = group_val_mask(ml, 0.03, 13)
ml_train, ml_val = ml[~l_val_mask].copy(), ml[l_val_mask].copy()
ml_val = ml_val[(ml_val.target <= 0.2) | (ml_val.target >= 0.8)].copy()
ml_val["target"] = (ml_val.target >= 0.5).astype(int)

for df in (m_train, m_val, ml_train, ml_val):
    df["category"] = [item_cat[i] for i in df.id1]
del m, ml, m_val_mask, l_val_mask
import gc; gc.collect()
print(f"manual: {len(m_train):,}/{len(m_val):,}  llm: {len(ml_train):,}/{len(ml_val):,}")

manual: 292,706/72,948  llm: 10,950,394/191,555


## 3. Dataset / метрика

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PairDS(Dataset):
    def __init__(self, pairs_df):
        self.id1 = pairs_df.id1.values
        self.id2 = pairs_df.id2.values
        self.y = pairs_df.target.values.astype(np.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        return item_text[self.id1[i]], item_text[self.id2[i]], self.y[i]

def collate(batch):
    t1, t2, y = zip(*batch)
    enc = tokenizer(list(t1), list(t2), padding=True, truncation=True,
                    max_length=MAX_LEN, return_tensors="pt")
    return enc, torch.tensor(y)

@torch.no_grad()
def predict(model, pairs_df, bs=512):
    model.eval()
    dl = DataLoader(PairDS(pairs_df), batch_size=bs, collate_fn=collate,
                    num_workers=0, shuffle=False)
    out = []
    for enc, _ in dl:
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.autocast(device_type="cuda", dtype=torch.float16):
            logits = model(**enc).logits.squeeze(-1)
        out.append(torch.sigmoid(logits.float()).cpu().numpy())
    return np.concatenate(out)

def macro_pr_auc(pairs_df, preds):
    z = pairs_df[["category", "target"]].copy(); z["pred"] = preds
    aps = z.groupby("category").apply(lambda g: average_precision_score(g.target, g.pred))
    return float(aps.mean()), aps

# быстрый eval-сэмпл, чтобы не гонять весь val каждые 20k шагов
ml_val_fast = ml_val.sample(min(60_000, len(ml_val)), random_state=0)

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

## 4. Стадия A — обучение на LLM-парах (soft labels)

In [5]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1).to(device)

def train_stage(pairs_df, epochs, lr, tag):
    dl = DataLoader(PairDS(pairs_df), batch_size=BATCH, collate_fn=collate,
                    num_workers=0, shuffle=True, drop_last=True)
    steps_total = len(dl) * epochs
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sched = get_linear_schedule_with_warmup(opt, min(WARMUP, steps_total // 10), steps_total)
    scaler = torch.amp.GradScaler()
    lossf = nn.BCEWithLogitsLoss()
    step, t0, run_loss = 0, time.time(), 0.0
    best = 0.0
    for ep in range(epochs):
        for enc, y in dl:
            model.train()
            enc = {k: v.to(device, non_blocking=True) for k, v in enc.items()}
            y = y.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                logits = model(**enc).logits.squeeze(-1)
                loss = lossf(logits, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sched.step()
            run_loss += loss.item(); step += 1
            if step % 2000 == 0:
                sps = step * BATCH / (time.time() - t0)
                print(f"[{tag}] ep{ep} step {step}/{steps_total} loss={run_loss/2000:.4f} {sps:.0f} pair/s", flush=True)
                run_loss = 0.0
            if step % EVAL_EVERY == 0:
                mac, _ = macro_pr_auc(ml_val_fast, predict(model, ml_val_fast))
                print(f"[{tag}] step {step}: llm-val-fast Macro PR-AUC = {mac:.4f}", flush=True)
                torch.save({"model": model.state_dict(), "step": step, "metric": mac}, CKPT)
                if mac > best: best = mac
    return best

train_stage(ml_train, EPOCHS_A, LR_A, "A/llm")

model.safetensors:   0%|          | 0.00/118M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider trai

[A/llm] ep0 step 2000/42774 loss=0.5064 605 pair/s
[A/llm] ep0 step 4000/42774 loss=0.4816 604 pair/s
[A/llm] ep0 step 6000/42774 loss=0.4641 604 pair/s
[A/llm] ep0 step 8000/42774 loss=0.4384 604 pair/s
[A/llm] ep0 step 10000/42774 loss=0.4217 604 pair/s
[A/llm] step 10000: llm-val-fast Macro PR-AUC = 0.6327


/tmp/ipykernel_23/3698110657.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  aps = z.groupby("category").apply(lambda g: average_precision_score(g.target, g.pred))


...

/tmp/ipykernel_23/3698110657.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  aps = z.groupby("category").apply(lambda g: average_precision_score(g.target, g.pred))


[A/llm] ep0 step 32000/42774 loss=0.3648 595 pair/s
[A/llm] ep0 step 34000/42774 loss=0.3616 595 pair/s
[A/llm] ep0 step 36000/42774 loss=0.3603 595 pair/s
[A/llm] ep0 step 38000/42774 loss=0.3597 596 pair/s
[A/llm] ep0 step 40000/42774 loss=0.3583 596 pair/s
[A/llm] step 40000: llm-val-fast Macro PR-AUC = 0.7457


/tmp/ipykernel_23/3698110657.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  aps = z.groupby("category").apply(lambda g: average_precision_score(g.target, g.pred))


[A/llm] ep0 step 42000/42774 loss=0.3549 594 pair/s


0.7457421527939506

## 5. Стадия B — дообучение на ручных парах

In [6]:
train_stage(m_train, EPOCHS_B, LR_B, "B/manual")

[B/manual] ep1 step 2000/2286 loss=0.3458 600 pair/s


0.0

## 6. Финальные метрики — сравнивать с GBM: llm 0.608 / manual 0.625

In [7]:
mac_l, aps_l = macro_pr_auc(ml_val, predict(model, ml_val))
print(f"[llm holdout, уверенные] Macro PR-AUC = {mac_l:.4f}")
print(aps_l.round(3).to_string())

mac_m, aps_m = macro_pr_auc(m_val, predict(model, m_val))
print(f"\n[manual holdout] Macro PR-AUC = {mac_m:.4f}")
print(aps_m.round(3).to_string())

/tmp/ipykernel_23/3698110657.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  aps = z.groupby("category").apply(lambda g: average_precision_score(g.target, g.pred))


[llm holdout, уверенные] Macro PR-AUC = 0.6851
category
Автотовары                 0.892
Аптека                     0.797
Бытовая техника            0.847
Бытовая химия              0.826
Галантерея и аксессуары    0.410
Детские товары             0.760
Дом и сад                  0.693
Канцелярские товары        0.688
Красота и гигиена          0.645
Мебель                     0.696
Музыкальные инструменты    0.860
Обувь                      0.194
Одежда                     0.260
Продукты питания           0.780
Спорт и отдых              0.712
Строительство и ремонт     0.789
Товары для животных        0.832
Хобби и творчество         0.783
Электроника                0.698
Ювелирные изделия          0.539

[manual holdout] Macro PR-AUC = 0.7208
category
Автотовары                 0.664
Аптека                     0.786
Бытовая техника            0.809
Бытовая химия              0.854
Галантерея и аксессуары    0.635
Детские товары             0.860
Дом и сад                  0.779
Канц

/tmp/ipykernel_23/3698110657.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  aps = z.groupby("category").apply(lambda g: average_precision_score(g.target, g.pred))


## 7. Сохранение — скачать папку целиком и прислать для сборки сабмита

In [8]:
os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR)
tokenizer.save_pretrained(OUT_DIR)
with open(f"{OUT_DIR}/metrics.json", "w") as fp:
    json.dump({"llm_holdout_macro_prauc": mac_l, "manual_holdout_macro_prauc": mac_m,
               "model": MODEL_NAME, "max_len": MAX_LEN,
               "llm_by_cat": aps_l.to_dict(), "manual_by_cat": aps_m.to_dict()}, fp,
              ensure_ascii=False, indent=1)
print("saved:", OUT_DIR)
!cd /kaggle/working && zip -qr ce_v1_final.zip ce_v1_final && du -h ce_v1_final.zip

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

saved: /kaggle/working/ce_v1_final
104M	ce_v1_final.zip
